# 06. Extractive Question Answering with BERT


## 1. What you will build

- A retrieval-by-doc-id + extractive QA flow with a BERT QA model.
- A measurable evaluation using exact match and token-level F1.
- A reusable function for enterprise policy-question workflows.


## 2. When to use this in real companies

Use this approach when users ask factual questions over policy, knowledge-base, SLA, or operational documents and you need span-level answers with confidence scores.


## 3. Business goal

Answer operational questions from policy documents accurately and transparently, with measurable quality metrics.


## 4. Imports and setup


In [1]:
from pathlib import Path
import re

import pandas as pd
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer


## 5. Load datasets from `data/06_data`


In [2]:
POLICIES_PATH = Path("../data/06_data/policies.csv")
QUESTIONS_PATH = Path("../data/06_data/qa_questions.csv")

if not POLICIES_PATH.exists():
    POLICIES_PATH = Path("data/06_data/policies.csv")
if not QUESTIONS_PATH.exists():
    QUESTIONS_PATH = Path("data/06_data/qa_questions.csv")

if not POLICIES_PATH.exists() or not QUESTIONS_PATH.exists():
    raise FileNotFoundError("Missing files in data/06_data")

kb = pd.read_csv(POLICIES_PATH)
questions = pd.read_csv(QUESTIONS_PATH)

print(f"Policies: {len(kb)}")
print(f"Questions: {len(questions)}")
kb.head()


Policies: 8
Questions: 13


,doc_id,context
0,policy_1,Customers can request a full refund within 14 ...
1,policy_2,Two-factor authentication is mandatory for adm...
2,policy_3,Standard delivery takes 3 to 5 business days. ...
3,policy_4,Critical incidents are acknowledged within 30 ...
4,policy_5,Enterprise contracts include a dedicated accou...


In [3]:
questions.head()


,question,doc_id,expected
0,How many days are allowed for full refunds?,policy_1,14 days
1,How long do refunds take to process?,policy_1,5 business days
2,Is 2FA required for admin users?,policy_2,mandatory
3,When does the reset link expire?,policy_2,30 minutes
4,What is standard delivery time?,policy_3,3 to 5 business days


## 6. Load QA model


In [4]:
class ExtractiveQAModel:
    """Compatibility wrapper that keeps qa(question=..., context=...) API."""

    def __init__(self, model_name: str, max_answer_len: int = 30):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.model.eval()
        self.max_answer_len = max_answer_len

    def __call__(self, question: str, context: str):
        inputs = self.tokenizer(
            question,
            context,
            return_tensors="pt",
            truncation="only_second",
            max_length=512,
        )

        with torch.no_grad():
            outputs = self.model(**inputs)

        start_logits = outputs.start_logits[0]
        end_logits = outputs.end_logits[0]
        start_probs = torch.softmax(start_logits, dim=-1)
        end_probs = torch.softmax(end_logits, dim=-1)

        best_score = float("-inf")
        best_start, best_end = 0, 0
        num_tokens = start_logits.size(0)

        for start_idx in range(num_tokens):
            end_limit = min(num_tokens - 1, start_idx + self.max_answer_len - 1)
            span_scores = start_logits[start_idx] + end_logits[start_idx : end_limit + 1]
            best_rel_end = int(torch.argmax(span_scores).item())
            score = float(span_scores[best_rel_end].item())
            if score > best_score:
                best_score = score
                best_start = start_idx
                best_end = start_idx + best_rel_end

        answer = self.tokenizer.decode(
            inputs["input_ids"][0][best_start : best_end + 1],
            skip_special_tokens=True,
        ).strip()

        confidence = float((start_probs[best_start] * end_probs[best_end]).item())
        return {"answer": answer, "score": confidence}


qa = ExtractiveQAModel("distilbert-base-cased-distilled-squad")


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

## 7. Ask/answer function


In [5]:
def ask_question(row):
    """Run extractive QA against the context selected by doc_id."""
    context = kb.loc[kb["doc_id"] == row["doc_id"], "context"].iloc[0]
    out = qa(question=row["question"], context=context)
    return pd.Series({"predicted": out["answer"], "confidence": out["score"]})

results = questions.join(questions.apply(ask_question, axis=1))
results.head()


,question,doc_id,expected,predicted,confidence
0,How many days are allowed for full refunds?,policy_1,14 days,14,0.699349
1,How long do refunds take to process?,policy_1,5 business days,5 business days,0.813616
2,Is 2FA required for admin users?,policy_2,mandatory,Two - factor authentication is mandatory,0.597203
3,When does the reset link expire?,policy_2,30 minutes,after 30 minutes,0.904088
4,What is standard delivery time?,policy_3,3 to 5 business days,3 to 5 business days,0.958660


## 8. Evaluation metrics


In [6]:
def normalize_text(x: str) -> str:
    return " ".join(re.sub(r"[^a-z0-9\s]", " ", str(x).lower()).split())


def token_f1(pred: str, gold: str) -> float:
    p = normalize_text(pred).split()
    g = normalize_text(gold).split()
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0

    common = set(p) & set(g)
    if not common:
        return 0.0

    precision = len(common) / len(p)
    recall = len(common) / len(g)
    return 2 * precision * recall / (precision + recall)


results["exact_match"] = results.apply(lambda r: normalize_text(r["predicted"]) == normalize_text(r["expected"]), axis=1)
results["token_f1"] = results.apply(lambda r: token_f1(r["predicted"], r["expected"]), axis=1)

results[["question", "expected", "predicted", "confidence", "exact_match", "token_f1"]].head(12)


,question,expected,predicted,confidence,exact_match,token_f1
0,How many days are allowed for full refunds?,14 days,14,0.699349,False,0.666667
1,How long do refunds take to process?,5 business days,5 business days,0.813616,True,1.000000
2,Is 2FA required for admin users?,mandatory,Two - factor authentication is mandatory,0.597203,False,0.333333
3,When does the reset link expire?,30 minutes,after 30 minutes,0.904088,False,0.800000
4,What is standard delivery time?,3 to 5 business days,3 to 5 business days,0.958660,True,1.000000
5,What is international delivery time?,up to 12 business days,12 business days,0.558523,False,0.750000
6,How quickly are critical incidents acknowledged?,30 minutes,within 30 minutes,0.712137,False,0.800000
7,How quickly are critical incidents resolved?,4 hours,within 4 hours,0.710220,False,0.800000
8,What is included in enterprise contracts?,a dedicated account manager and quarterly arch...,a dedicated account manager and quarterly arch...,0.474238,True,1.000000
9,When are invoices issued?,the first day of each month,on the first day of each month,0.468497,False,0.923077


In [7]:
pd.Series(
    {
        "exact_match": results["exact_match"].mean(),
        "avg_token_f1": results["token_f1"].mean(),
        "avg_confidence": results["confidence"].mean(),
    }
).round(3)


exact_match       0.462
avg_token_f1      0.852
avg_confidence    0.678
dtype: float64

## 9. Production-style helper


In [8]:
def answer_question(question: str, doc_id: str):
    """Return one QA answer payload for a selected policy document."""
    context = kb.loc[kb["doc_id"] == doc_id, "context"].iloc[0]
    out = qa(question=question, context=context)
    return {
        "doc_id": doc_id,
        "question": question,
        "answer": out["answer"],
        "confidence": round(float(out["score"]), 4),
    }

answer_question("How often are API credentials rotated?", "policy_7")


{'doc_id': 'policy_7',
 'question': 'How often are API credentials rotated?',
 'answer': 'every 90 days',
 'confidence': 0.5343}

## 10. Summary

- Data is externalized under `data/06_data` for reproducibility.
- The notebook includes both prediction flow and measurable QA quality.
- This structure is suitable for enterprise policy assistants and internal knowledge bots.
